In [2]:
import polars as pl

In [3]:
! pwd

/home/phinguyen/de_upskills_mock_prj


In [4]:
! ls data/MovieLens

genome_scores.csv  genome_tags.csv  link.csv  movie.csv  rating.csv  tag.csv


# Movie

In [5]:
movie = pl.read_csv("data/MovieLens/movie.csv")

In [6]:
movie

movieId,title,genres
i64,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…"
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance"""
5,"""Father of the Bride Part II (1…","""Comedy"""
…,…,…
131254,"""Kein Bund für's Leben (2007)""","""Comedy"""
131256,"""Feuer, Eis & Dosenbier (2002)""","""Comedy"""
131258,"""The Pirates (2014)""","""Adventure"""


In [7]:
# Check unique values in the "genres" column
movie.select(
    pl.col("genres").unique().alias("unique_genres")
)

unique_genres
str
"""Adventure|Drama|Mystery"""
"""Horror|Sci-Fi|Thriller"""
"""Adventure|Musical"""
"""Action|Comedy|Fantasy|IMAX"""
"""Action|Documentary|Drama|Roman…"
…
"""Action|Adventure|Fantasy"""
"""Action|Crime|Film-Noir"""
"""Children|Comedy|Drama|Musical"""


In [8]:
# sample of genres is "Comedy|Horror|Mystery|Sci-Fi"
# transform the genres column to rows, for example, "Comedy|Horror|Mystery|Sci-Fi" will be transformed to 4 rows: "Comedy", "Horror", "Mystery", "Sci-Fi"

movie_exploded = movie.with_columns(
    pl.col("genres").str.split("|").alias("genres_list")
).explode("genres_list").rename({"genres_list": "genre"})

movie_exploded

/tmp/ipykernel_6047/3391097171.py:6: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  ).explode("genres_list").rename({"genres_list": "genre"})


movieId,title,genres,genre
i64,str,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Adventure"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Animation"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Children"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Comedy"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Fantasy"""
…,…,…,…
131258,"""The Pirates (2014)""","""Adventure""","""Adventure"""
131260,"""Rentun Ruusu (2001)""","""(no genres listed)""","""(no genres listed)"""
131262,"""Innocence (2014)""","""Adventure|Fantasy|Horror""","""Adventure"""


In [9]:
movie_exploded.select(
    pl.col("genre").unique().alias("unique_genre")
).write_csv("data/investigation/movie_unique_genres.csv")

In [10]:
movie = movie.with_columns(
    pl.col("title")
    .str.extract(r"^(.*) \((\d{4})\)$", group_index=1)
    .alias("movie_name"),
    pl.col("title")
    .str.extract(r"^(.*) \((\d{4})\)$", group_index=2)
    .cast(pl.Int32)
    .alias("year"),
)

movie

movieId,title,genres,movie_name,year
i64,str,str,str,i32
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Toy Story""",1995
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy""","""Jumanji""",1995
3,"""Grumpier Old Men (1995)""","""Comedy|Romance""","""Grumpier Old Men""",1995
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance""","""Waiting to Exhale""",1995
5,"""Father of the Bride Part II (1…","""Comedy""","""Father of the Bride Part II""",1995
…,…,…,…,…
131254,"""Kein Bund für's Leben (2007)""","""Comedy""","""Kein Bund für's Leben""",2007
131256,"""Feuer, Eis & Dosenbier (2002)""","""Comedy""","""Feuer, Eis & Dosenbier""",2002
131258,"""The Pirates (2014)""","""Adventure""","""The Pirates""",2014


In [11]:
year_pattern = r"^(.*) \((\d{4})\)$"

unmatched_titles = (
    movie
    .with_columns(
        pl.col("title")
        .str.extract(year_pattern, group_index=1)
        .alias("movie_name"),
        pl.col("title")
        .str.extract(year_pattern, group_index=2)
        .cast(pl.Int32)
        .alias("year"),
    )
    .filter(pl.col("year").is_null())
    .select("title", "movie_name", "year")
)

unmatched_titles.write_csv("data/investigation/unmatched_titles.csv")

In [12]:
year_pattern = r"^(.*?)[\s\u00A0]*\((\d{4})\)$"

parsed = movie.with_columns(
    pl.col("title")
    .str.strip_chars()
    .str.extract(year_pattern, group_index=1)
    .alias("movie_name"),
    pl.col("title")
    .str.strip_chars()
    .str.extract(year_pattern, group_index=2)
    .cast(pl.Int32)
    .alias("year"),
)

unmatched_titles = parsed.filter(pl.col("year").is_null())

In [13]:
unmatched_titles.write_csv("data/investigation/unmatched_titles2.csv")

In [14]:
import polars as pl

title = pl.col("title").str.strip_chars()

# Accept a terminal year with normal spacing, or one extra closing parenthesis.
year_text = title.str.extract(r"\((\d{4})\)\)?$", group_index=1)

# TV-style ranges are flagged separately; don't treat their start year as movie release year.
has_year_range = title.str.contains(r"\(\d{4}\s*[-–]\s*\d{4}?\s*\)$")

result = movie.with_columns(
    title.str.replace(r"\s*\(\d{4}\)\)?$", "").alias("movie_name"),
    year_text.cast(pl.Int32).alias("year"),
    has_year_range.fill_null(False).alias("has_year_range"),
).with_columns(
    pl.when(pl.col("year").is_not_null())
      .then(pl.lit("parsed"))
      .when(pl.col("has_year_range"))
      .then(pl.lit("year_range_review"))
      .otherwise(pl.lit("year_missing"))
      .alias("year_parse_status")
)

In [15]:
result

movieId,title,genres,movie_name,year,has_year_range,year_parse_status
i64,str,str,str,i32,bool,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Toy Story""",1995,false,"""parsed"""
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy""","""Jumanji""",1995,false,"""parsed"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance""","""Grumpier Old Men""",1995,false,"""parsed"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance""","""Waiting to Exhale""",1995,false,"""parsed"""
5,"""Father of the Bride Part II (1…","""Comedy""","""Father of the Bride Part II""",1995,false,"""parsed"""
…,…,…,…,…,…,…
131254,"""Kein Bund für's Leben (2007)""","""Comedy""","""Kein Bund für's Leben""",2007,false,"""parsed"""
131256,"""Feuer, Eis & Dosenbier (2002)""","""Comedy""","""Feuer, Eis & Dosenbier""",2002,false,"""parsed"""
131258,"""The Pirates (2014)""","""Adventure""","""The Pirates""",2014,false,"""parsed"""


# Source schema and CDM profiling

This section gathers the evidence needed before defining the lakehouse contracts. The assignment asks you to profile what is actually present; its listed column types and relationships are expectations to validate, not values to assume.

For each file, capture: (1) source columns and inferred physical types; (2) row count, null/blank counts, ranges and representative values; (3) candidate key uniqueness and grain; (4) domain and relationship checks; and (5) meaning, timestamp semantics, and whether a field is source-entered or derived. These findings inform a raw-preserving Bronze schema, a typed/conformed Silver CDM, and Gold facts/dimensions. Avoid changing raw source data during profiling.

## 1. Actual schema, nulls, value ranges, and examples

The following lazy profiler scans each source and writes two reviewable outputs under `data/investigation/`. It does not calculate exact cardinality for every column (which would be costly for 20M-row identifiers); key uniqueness is checked separately below.

In [16]:
from pathlib import Path
import polars as pl

DATA_DIR = Path("data/MovieLens")
OUT_DIR = Path("data/investigation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {
    "rating": "rating.csv",
    "tag": "tag.csv",
    "movie": "movie.csv",
    "link": "link.csv",
    "genome_scores": "genome_scores.csv",
    "genome_tags": "genome_tags.csv",
}

def source_scan(file_name: str) -> pl.LazyFrame:
    # Keep the source timestamp representation as-is for this physical schema profile.
    return pl.scan_csv(DATA_DIR / file_name, infer_schema_length=10_000, try_parse_dates=False, null_values=[""])

source_frames = {name: source_scan(file_name) for name, file_name in SOURCE_FILES.items()}

schema_rows = []
column_rows = []
for table_name, lf in source_frames.items():
    print(f"Scanning {table_name}...")
    schema = lf.collect_schema()
    # One aggregate per source: total rows and null counts in a single scan.
    null_exprs = [pl.col(c).null_count().alias(f"null__{c}") for c in schema.names()]
    summary = lf.select([pl.len().alias("row_count"), *null_exprs]).collect().to_dicts()[0]
    for column_name, dtype in schema.items():
        schema_rows.append({
            "table": table_name, "source_file": SOURCE_FILES[table_name],
            "column": column_name, "polars_inferred_type": str(dtype),
            "row_count": summary["row_count"],
            "null_count": summary[f"null__{column_name}"],
        })
        expr = pl.col(column_name)
        # Min/max aid numeric/date range checks; for strings they are lexical, so treat as hints only.
        stats = lf.select(
            expr.min().cast(pl.String).alias("min_value"),
            expr.max().cast(pl.String).alias("max_value"),
            expr.drop_nulls().head(5).cast(pl.String).alias("sample_values"),
        ).collect().to_dicts()[0]
        column_rows.append({
            "table": table_name, "column": column_name,
            "min_value": stats["min_value"], "max_value": stats["max_value"],
            "sample_values": stats["sample_values"],
        })

schema_profile = pl.DataFrame(schema_rows).sort("table", "column")
column_profile = pl.DataFrame(column_rows).sort("table", "column")
schema_profile.write_csv(OUT_DIR / "schema_profile.csv")
column_profile.write_csv(OUT_DIR / "column_value_profile.csv")
schema_profile, column_profile


Scanning rating...
Scanning tag...
Scanning movie...
Scanning link...
Scanning genome_scores...
Scanning genome_tags...


(shape: (19, 6)
 ┌───────────────┬───────────────────┬───────────┬──────────────────────┬───────────┬────────────┐
 │ table         ┆ source_file       ┆ column    ┆ polars_inferred_type ┆ row_count ┆ null_count │
 │ ---           ┆ ---               ┆ ---       ┆ ---                  ┆ ---       ┆ ---        │
 │ str           ┆ str               ┆ str       ┆ str                  ┆ i64       ┆ i64        │
 ╞═══════════════╪═══════════════════╪═══════════╪══════════════════════╪═══════════╪════════════╡
 │ genome_scores ┆ genome_scores.csv ┆ movieId   ┆ Int64                ┆ 11709768  ┆ 0          │
 │ genome_scores ┆ genome_scores.csv ┆ relevance ┆ Float64              ┆ 11709768  ┆ 0          │
 │ genome_scores ┆ genome_scores.csv ┆ tagId     ┆ Int64                ┆ 11709768  ┆ 0          │
 │ genome_tags   ┆ genome_tags.csv   ┆ tag       ┆ String               ┆ 1128      ┆ 0          │
 │ genome_tags   ┆ genome_tags.csv   ┆ tagId     ┆ Int64                ┆ 1128      ┆ 0      

## 2. Confirm candidate keys and table grain

A grain is what one row represents. Check the expected source keys empirically. Ratings and tags are event tables: repeated user/movie combinations can be legitimate, so exact row duplication and candidate-key behavior are separate checks. Genome scores should have one row per `(movieId, tagId)`.

In [17]:
# Key profile: reports row count, distinct key count, and duplicate excess.
# Exact distinct counts on these small composite key sets are intentional full-data checks.
KEYS = {
    "rating": ["userId", "movieId"],
    "tag": ["userId", "movieId", "tag", "timestamp"],
    "movie": ["movieId"],
    "link": ["movieId"],
    "genome_scores": ["movieId", "tagId"],
    "genome_tags": ["tagId"],
}

key_rows = []
for table_name, key_columns in KEYS.items():
    lf = source_frames[table_name]
    key_stats = lf.select(
        pl.len().alias("row_count"),
        pl.struct(key_columns).n_unique().alias("distinct_key_count"),
        pl.struct(key_columns).is_duplicated().sum().alias("rows_marked_duplicate"),
    ).collect().to_dicts()[0]
    key_rows.append({
        "table": table_name, "candidate_key": ", ".join(key_columns),
        **key_stats,
        "row_count_minus_distinct_keys": key_stats["row_count"] - key_stats["distinct_key_count"],
    })
key_profile = pl.DataFrame(key_rows)
key_profile.write_csv(OUT_DIR / "key_profile.csv")
key_profile


table,candidate_key,row_count,distinct_key_count,rows_marked_duplicate,row_count_minus_distinct_keys
str,str,i64,i64,i64,i64
"""rating""","""userId, movieId""",20000263,20000263,0,0
"""tag""","""userId, movieId, tag, timestam…",465564,465564,0,0
"""movie""","""movieId""",27278,27278,0,0
"""link""","""movieId""",27278,27278,0,0
"""genome_scores""","""movieId, tagId""",11709768,11709768,0,0
"""genome_tags""","""tagId""",1128,1128,0,0


## 3. Type/domain checks and relationship evidence

Use the results to choose explicit Silver types and DQ rules. In particular, validate timestamp parsing and event ranges, rating increments/range, genome relevance range, IMDb/TMDb nullability, and foreign keys. Raw strings remain available in Bronze even when parsing fails.

In [18]:
# Parse event timestamps as a derived profiling expression. Source CSV columns remain unchanged.
def with_event_time(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        pl.col("timestamp").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("event_time")
    )

rating_checks = with_event_time(source_frames["rating"]).select(
    pl.len().alias("rows"),
    pl.col("rating").null_count().alias("rating_nulls"),
    ((pl.col("rating") < 0.5) | (pl.col("rating") > 5.0) | ((pl.col("rating") * 2) % 1 != 0)).fill_null(False).sum().alias("invalid_rating_rows"),
    pl.col("event_time").null_count().alias("timestamp_parse_failures"),
    pl.col("event_time").min().alias("event_time_min"),
    pl.col("event_time").max().alias("event_time_max"),
).collect()

tag_checks = with_event_time(source_frames["tag"]).select(
    pl.len().alias("rows"),
    pl.col("tag").null_count().alias("null_tags"),
    (pl.col("tag").str.strip_chars().fill_null("") == "").sum().alias("empty_or_whitespace_tags"),
    pl.col("event_time").null_count().alias("timestamp_parse_failures"),
).collect()

genome_score_checks = source_frames["genome_scores"].select(
    pl.len().alias("rows"),
    pl.col("relevance").null_count().alias("null_relevance"),
    ((pl.col("relevance") < 0) | (pl.col("relevance") > 1)).fill_null(False).sum().alias("out_of_range_relevance"),
).collect()

movie_checks = source_frames["movie"].select(
    pl.len().alias("rows"),
    pl.col("title").null_count().alias("null_titles"),
    pl.col("genres").null_count().alias("null_genres"),
    (pl.col("genres") == "(no genres listed)").sum().alias("no_genres_sentinel_rows"),
).collect()

link_checks = source_frames["link"].select(
    pl.len().alias("rows"),
    pl.col("imdbId").null_count().alias("null_imdb_ids"),
    pl.col("tmdbId").null_count().alias("null_tmdb_ids"),
).collect()
rating_checks, tag_checks, movie_checks, genome_score_checks, link_checks


(shape: (1, 6)
 ┌──────────┬──────────────┬──────────────────┬─────────────────┬─────────────────┬─────────────────┐
 │ rows     ┆ rating_nulls ┆ invalid_rating_r ┆ timestamp_parse ┆ event_time_min  ┆ event_time_max  │
 │ ---      ┆ ---          ┆ ows              ┆ _failures       ┆ ---             ┆ ---             │
 │ u32      ┆ u32          ┆ ---              ┆ ---             ┆ datetime[μs]    ┆ datetime[μs]    │
 │          ┆              ┆ u32              ┆ u32             ┆                 ┆                 │
 ╞══════════╪══════════════╪══════════════════╪═════════════════╪═════════════════╪═════════════════╡
 │ 20000263 ┆ 0            ┆ 0                ┆ 0               ┆ 1995-01-09      ┆ 2015-03-31      │
 │          ┆              ┆                  ┆                 ┆ 11:46:44        ┆ 06:40:02        │
 └──────────┴──────────────┴──────────────────┴─────────────────┴─────────────────┴─────────────────┘,
 shape: (1, 4)
 ┌────────┬───────────┬──────────────────────────┬─

In [19]:
# Referential integrity anti-joins. These can take time because the large files are scanned.
def anti_join_count(child: pl.LazyFrame, child_key: str, parent: pl.LazyFrame, parent_key: str) -> int:
    return child.join(
        parent.select(pl.col(parent_key).alias("_parent_key")).unique(),
        left_on=child_key, right_on="_parent_key", how="anti",
    ).select(pl.len()).collect().item()

relationship_profile = pl.DataFrame({
    "relationship": [
        "rating.movieId -> movie.movieId", "tag.movieId -> movie.movieId",
        "link.movieId -> movie.movieId", "movie.movieId -> link.movieId",
        "genome_scores.movieId -> movie.movieId", "genome_scores.tagId -> genome_tags.tagId",
    ],
    "orphan_rows": [
        anti_join_count(source_frames["rating"], "movieId", source_frames["movie"], "movieId"),
        anti_join_count(source_frames["tag"], "movieId", source_frames["movie"], "movieId"),
        anti_join_count(source_frames["link"], "movieId", source_frames["movie"], "movieId"),
        anti_join_count(source_frames["movie"], "movieId", source_frames["link"], "movieId"),
        anti_join_count(source_frames["genome_scores"], "movieId", source_frames["movie"], "movieId"),
        anti_join_count(source_frames["genome_scores"], "tagId", source_frames["genome_tags"], "tagId"),
    ],
})
relationship_profile.write_csv(OUT_DIR / "relationship_profile.csv")
relationship_profile


relationship,orphan_rows
str,i64
"""rating.movieId -> movie.movieI…",0
"""tag.movieId -> movie.movieId""",0
"""link.movieId -> movie.movieId""",0
"""movie.movieId -> link.movieId""",0
"""genome_scores.movieId -> movie…",0
"""genome_scores.tagId -> genome_…",0


## 4. Turn profiles into model decisions

Use measured output rather than assumptions to complete a mapping document. A starting model hypothesis (to validate, not blindly adopt):

- **Rating event:** one row per rating interaction; preserve `rating` on the source 0.5–5 scale, record normalized event time in UTC, and keep source system/key lineage. Do not assume `(userId, movieId)` is unique until the key profile confirms the intended semantics.
- **User tag event:** one row per tag submission; preserve raw text and create a separately normalized tag value in Silver. The key may need timestamp or a source row identifier because users can repeat tags.
- **Movie:** conformed content entity keyed by MovieLens `movieId`; split title, extracted release year, genre bridge rows, and external IDs into suitable Silver/Gold structures. Retain the source title and parsing status.
- **Genome score:** derived movie–genome-tag measurement, grain `(movieId, tagId)`; keep distinct from user-submitted tags.
- **Link:** movie identifier crosswalk; test the actual one-to-one coverage and permit missing external IDs where observed.
- **Bronze:** append-only, source-shaped typed data plus batch/file/hash/ingestion audit fields; do not cleanse or drop malformed values here.
- **Silver:** explicit types, normalized/conformed names and units, deterministic deduplication, DQ quarantine, and incremental MERGE. Add derived columns without overwriting raw source values.
- **Gold:** declare fact grain and use surrogate-key dimensions. Movie genre history needs Type 2 if point-in-time genre analysis is required; decide Type 1/3 per attribute based on analytical need.

Save your actual schema, key, relationship, and domain outputs alongside a short explanation of any deviations from the assignment's expected counts/types.